In [2]:
from  importlib.metadata import version
print("torch version" , version("torch"))


torch version 2.11.0+cpu


In [3]:
import torch
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

Step 1: compute unnormalized attention scores  𝜔
Suppose we use the second input token as the query, that is,  𝑞(2)=𝑥(2) , we compute the unnormalized attention scores via dot products:
𝜔21=𝑥(1)𝑞(2)⊤
𝜔22=𝑥(2)𝑞(2)⊤
𝜔23=𝑥(3)𝑞(2)⊤
...
𝜔2𝑇=𝑥(𝑇)𝑞(2)⊤

In [4]:
query = inputs[1] # here we take 2nd input token   as the query first
attn_scores_2 = torch.empty(inputs.shape[0])
print(attn_scores_2)
for i , x_i in enumerate(inputs):
  attn_scores_2[i]=torch.dot(x_i , query)#here we are not taking the transpose   bcz they are 1 D vectora

print(attn_scores_2)



tensor([2.3086e-12, 7.2646e+22, 7.2250e+28, 1.5766e-19, 6.3376e-10, 2.1707e-18])
tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


demonstrate dot poduct :=>essentially a shorthand for multiplying two vectors elements-wise and summing the resulting products:


In [5]:
res = 0.

for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

print(res)
print(torch.dot(inputs[0], query))

tensor(0.9544)
tensor(0.9544)


- **Step 2:** normalize the unnormalized attention scores ("omegas", $\omega$) so that they sum up to 1
- Here is a simple way to normalize the unnormalized attention scores to sum up to 1 (a convention, useful for interpretation, and important for training stability):

In [6]:
attn_weights_2_tmp=attn_scores_2/attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)



=> but in practise we use softmax function for normalization , which is better at handling  extreme values and has more desirable gradient  properties  during training , ( it is used mostly  and recommed for  use)


\

In [7]:
#defining softmax function
def softmax_naive(x):
  return torch.exp(x)/torch.exp(x).sum(dim=0)




In [8]:
attn_weights_2_naive = softmax_naive(attn_scores_2)

print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


sum is 1 . so it is confirmed that  the attention  score become attention\ weights

the above softmax is our raw method to get you and overview about  how does softmax normalize  the attention scores \
in practice, it's recommended to use the PyTorch implementation of softmax instead, which has been highly optimized for performance:\\

In [9]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


- **Step 3**: compute the context vector $z^{(2)}$ by multiplying the embedded input tokens, $x^{(i)}$ with the attention weights and sum the resulting vectors:


In [10]:
query = inputs[1] # 2nd input token is the query

context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


i=>n today session we learnt about  self attention .

=> why we use dot product for calulating  the attention score.

=> diff between attention score and attention weight

=>why do  we normalize   the attention score to foam  attention weight.

==>how to compute context vector z(i)

==> underhoood working  of softmax  method provided by torch  library
 \
from tommarow i will start 3.3.2  

- **Step 3**: compute the context vector $z^{(2)}$ by multiplying the embedded input tokens, $x^{(i)}$ with the attention weights and sum the resulting vectors:


In [11]:
query = inputs[1] # 2nd input token is the query

context_vec_2 = torch.zeros(query.shape)
print("context vetor initially ", context_vec_2)

print(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i
    print (context_vec_2)

print(context_vec_2)

context vetor initially  tensor([0., 0., 0.])
torch.Size([3])
tensor([0.0596, 0.0208, 0.1233])
tensor([0.1904, 0.2277, 0.2803])
tensor([0.3234, 0.4260, 0.4296])
tensor([0.3507, 0.4979, 0.4705])
tensor([0.4340, 0.5250, 0.4813])
tensor([0.4419, 0.6515, 0.5683])
tensor([0.4419, 0.6515, 0.5683])




3.3.2 Computing attention weights for all input tokens **bold text**

#### Generalize to all input sequence tokens:

- Above, we computed the attention weights and context vector for input 2 (as illustrated in the highlighted row in the figure below)
- Next, we are generalizing this computation to compute all attention weights and context vectors


In [12]:
attn_scores = torch.empty(6, 6)
#here we are computing the attention  scores of all the input tokens
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


we can achieve the same by doing  matrix multiplication  because it  is  more convinient and concise


In [ ]:
attn_scores = inputs @ inputs.T
print(attn_scores)

now normalizing it to  produce attention weight through  inbuilt6 softmax function

In [13]:
attn_weights = torch.softmax(attn_scores, dim=1)
print(attn_weights)



tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


now computing the context  vectors to all the input weights


In [14]:
all_context_vectors = attn_weights@inputs
print(all_context_vectors)


tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


&nbsp;
## 3.4 Implementing self-attention with trainable weights

- A conceptual framework illustrating how the self-attention mechanism developed in this section integrates into the overall narrative and structure of this book and chapter

thats it for today  , today i learn about how to compute the context vector from the ground up  , and validates it  ,  with what   we gets from earliar z_2 context vectors